# function $e^x$:

**Exercise 1.9**

**(a)** Write a program to compute the exponential function $e^x$ using the infinite series

$$
e^x=1+x+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

**(b)** Summing in the natural order, what stopping criterion should you use?

**(c)** Test your program for

$$
x=\pm1,\ \pm5,\ \pm10,\ \pm15,\ \pm20
$$

and compare your results with the built-in function

$$
\exp(x)
$$

**(d)** Can you use the series in this form to obtain accurate results for $x<0$?

**Hint:**

$$
e^{-x}=\frac{1}{e^x}
$$

**(e)** Can you rearrange the series or regroup the terms in any way to obtain more accurate results for $x<0$?

---

Este notebook calcula y compara el eror para el valor "real" y aproximado de $e^x$.

## 1. Setup
Usaremos la librería `math` para calcular el exponente "real" y `pandas` para organizar los resultados en una tabla. 

In [1]:
import math
import pandas as pd

## 2. Definición de variables, ¿Cómo obtendremos cada una? 

- **Valor exacto:** $e^x$, cáculado con el método definido de math:  `math.exp()`.
- **$e^x$ aproximation:** Usando la fórmula dada: $e^x=1+x+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots$

Y las ya conocidas: 
- **Absolute error:** $E_a = |e - e_aprox|$.
- **Relative error:** $r = E_a / e$

### b) ¿Cuántos términos son suficientes para obtener un valor apropiado de e^x? 

Podríamos pensar que entre más mejor, pero si recordamos lo que ha pasado con las aproximaciones de stirling y $e$ se debe tener cuidado con los cocientes y los factoriales. Pasan dos detalles, como vimos en stirling el factorial crece mucho más rápido que el exponente, y al ser el denominador llegará el punto en el que el cociente represente valores muy pequeños. ahora, ¿de verdad tiene efecto sumar valores cercanos a cero?. Además, si agregamos otro detalle más, el encontrarse calculando en un entorno computacional ya pone un límite, con la doble precision tenemos solo 16 dígitpos de precisión y con simple 8 dígitos, cortados por redondeo al más cercano. Llegará un punto en el que simplemente se redondee a cero y el aporte sea nulo. 

Ya existe una pista, si n tiene más de 16 dígitos no sería significativo, pero esto pasa solo para mayores a 1, pues si x es por ejemplo, 0.00001, que nesecitaría más términos para cubrir la precisión. 

El valor n debe ser relativo al valor de x. Definiendo la tolerancia 10^15, se realizarán iteraciones bucles que comparan el valor que se acaba de sumar a la serie con el valor absoluto actual. Si es menor a la toleracia, se acaba la iteración de la serie. 

In [17]:
## Definición de funciones: cómputo aproximado de e^x y cálculo de errores
## nombramos n al valor de los exponentes de e^x que se van a calcular, es decir, n = 0, 1, 2, ..., n-1
## Se usarará la función math.factorial() para calcular el factorial de los exponentes.

def compute_taylor_sum(x, tol=1e-15, max_iter=1000):
    total = 0.0
    i = 0
    while True:
        term = (x ** i) / math.factorial(i)
        total += term
        if abs(term) < tol * abs(total) or i >= max_iter:
            break
        i += 1
    return total, i + 1   

def get_error_values(x):
    exact = math.exp(x)
    approx, _ = compute_taylor_sum(x)
    abs_err = abs(exact - approx)
    rel_err = abs_err / exact
    return exact, approx, abs_err, rel_err

## 3. Cálculo y visualización de resultados

In [19]:
## c)  Test your program for [-1, 1, -5, 5, -10, 10, -15, 15, -20, 20]

rows = []
define_x_values = [-1, 1, -5, 5, -10, 10, -15, 15, -20, 20]  # Valores de x para los cuales se calculará e^x   

for k in range(len(define_x_values)):
    x = define_x_values[k]
    
    exact, approx, abs_err, rel_err = get_error_values(x)
    rows.append({
        "x": x,
        "e": exact,
        "e_aprox": round(approx, 16),
        "Abs. Error": round(abs_err, 16),
        "Rel. Error": rel_err,
    })

df = pd.DataFrame(rows)
df

,x,e,e_aprox,Abs. Error,Rel. Error
0,-1,3.678794e-01,3.678794e-01,1.000000e-16,3.017899e-16
1,1,2.718282e+00,2.718282e+00,4.000000e-16,1.633713e-16
2,-5,6.737947e-03,6.737947e-03,1.400000e-15,2.135596e-13
3,5,1.484132e+02,1.484132e+02,2.840000e-14,1.915040e-16
4,-10,4.539993e-05,4.539993e-05,3.289000e-13,7.244001e-09
5,10,2.202647e+04,2.202647e+04,7.276000e-12,3.303280e-16
6,-15,3.059023e-07,3.059342e-07,3.191610e-11,1.043344e-04
7,15,3.269017e+06,3.269017e+06,4.656613e-10,1.424469e-16
8,-20,2.061154e-09,5.478103e-10,1.513343e-09,7.342215e-01
9,20,4.851652e+08,4.851652e+08,2.980232e-07,6.142716e-16


## 4. Observaciones: 

- **Valores $x > 0$:** El error relativo se mantiene estable, todos con un valor cercano a 10^-16, tal como la precisión de la máquina. 

- **Valores $x < 0$:** Acá la situación desmejora mucho, con más lejano a cero el valor negativo, mayor error, incluso para x = -20 se captura un error relativo del 73%. Esto definitivamente señala lo sensible que es la aproximación a valores negativos.

### d) ¿Se pueden obtener valores bien aproximados para x < 0? 

Con las observaciones anteirores definitivamente no! y esto es porque los términos de la serie terminan con el signo alterado. Para x=−20 se cancelan números grandes (∼108) para dar un resultado diminuto (∼10−9), y el redondeo de esa cancelación es mayor que el resultado mismo.

In [28]:
## mejorando la función para que pueda aceptar valores negativos de x, usando la propiedad e^(-x) = 1 / e^x
def compute_taylor_sum_refactored(x, tol=1e-15, max_iter=1000):
    if x < 0:
        approx_pos, n = compute_taylor_sum(-x, tol, max_iter)
        return 1.0 / approx_pos, n
    else:
        return compute_taylor_sum(x, tol, max_iter)

def get_error_values(x):
    exact = math.exp(x)
    approx, _ = compute_taylor_sum_refactored(x)
    abs_err = abs(exact - approx)
    rel_err = abs_err / exact
    return exact, approx, abs_err, rel_err

In [29]:
rows = []
define_x_values = [-1, -5, -10, -15, -20,]   

for k in range(len(define_x_values)):
    x = define_x_values[k]
    
    exact, approx, abs_err, rel_err = get_error_values(x)
    rows.append({
        "x": x,
        "e": exact,
        "e_aprox": round(approx, 16),
        "Abs. Error": round(abs_err, 16),
        "Rel. Error": rel_err,
    })

df = pd.DataFrame(rows)
df

,x,e,e_aprox,Abs. Error,Rel. Error
0,-1,3.678794e-01,3.678794e-01,1.000000e-16,1.508950e-16
1,-5,6.737947e-03,6.737947e-03,0.000000e+00,2.574558e-16
2,-10,4.539993e-05,4.539993e-05,0.000000e+00,1.492571e-16
3,-15,3.059023e-07,3.059023e-07,0.000000e+00,1.730603e-16
4,-20,2.061154e-09,2.061154e-09,0.000000e+00,6.019789e-16


#### e) Se comprueba que con la segunda propuesta de aproximación para números negativos el error relativo consigue estabilizarse a valores mínimos como los obtenidos con valores de x positivos. 